In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

input_path = Path(r"C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed\train_full_anthropometry_audit_v1.csv")

df = pd.read_csv(input_path)

print(df.shape)
df.head()

(80000, 93)


,patient_id,site_id,triage_nurse_id,arrival_mode,arrival_hour,arrival_day,arrival_month,arrival_season,shift,age,...,height_implausible_flag,weight_implausible_flag,bmi_implausible_flag,is_pediatric,is_adult,anthro_error_likely_flag,anthro_extreme_height_flag,anthro_bmi_boundary_flag,anthro_review_flag,anthro_quality_category
0,TG-UXRGA9UCO,SITE-TMP-01,NURSE-0033,walk-in,6,Monday,5,spring,morning,43,...,0,0,0,0,1,0,0,0,0,00_no_obvious_anthro_error
1,TG-B19DBBS2G,SITE-HEL-01,NURSE-0001,walk-in,6,Thursday,4,spring,morning,72,...,0,0,0,0,1,0,0,0,0,00_no_obvious_anthro_error
2,TG-GZ97W7M6V,SITE-HEL-02,NURSE-0005,walk-in,8,Saturday,4,spring,morning,82,...,0,0,0,0,1,0,0,0,0,00_no_obvious_anthro_error
3,TG-THIB2TN9Q,SITE-HEL-02,NURSE-0026,police,7,Sunday,3,spring,morning,50,...,0,0,0,0,1,0,0,0,0,00_no_obvious_anthro_error
4,TG-J3U3LQ2QY,SITE-HEL-02,NURSE-0044,walk-in,5,Tuesday,5,spring,night,62,...,0,0,0,0,1,0,0,0,0,00_no_obvious_anthro_error


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 93 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   patient_id                      80000 non-null  object 
 1   site_id                         80000 non-null  object 
 2   triage_nurse_id                 80000 non-null  object 
 3   arrival_mode                    80000 non-null  object 
 4   arrival_hour                    80000 non-null  int64  
 5   arrival_day                     80000 non-null  object 
 6   arrival_month                   80000 non-null  int64  
 7   arrival_season                  80000 non-null  object 
 8   shift                           80000 non-null  object 
 9   age                             80000 non-null  int64  
 10  age_group                       80000 non-null  object 
 11  sex                             80000 non-null  object 
 12  language                        

In [4]:
df.columns.tolist()

['patient_id',
 'site_id',
 'triage_nurse_id',
 'arrival_mode',
 'arrival_hour',
 'arrival_day',
 'arrival_month',
 'arrival_season',
 'shift',
 'age',
 'age_group',
 'sex',
 'language',
 'insurance_type',
 'transport_origin',
 'pain_location',
 'mental_status_triage',
 'chief_complaint_system',
 'num_prior_ed_visits_12m',
 'num_prior_admissions_12m',
 'num_active_medications',
 'num_comorbidities',
 'systolic_bp',
 'diastolic_bp',
 'mean_arterial_pressure',
 'pulse_pressure',
 'heart_rate',
 'respiratory_rate',
 'temperature_c',
 'spo2',
 'gcs_total',
 'pain_score',
 'weight_kg',
 'height_cm',
 'bmi',
 'shock_index',
 'news2_score',
 'disposition',
 'ed_los_hours',
 'triage_acuity',
 'chief_complaint_raw',
 'hx_hypertension',
 'hx_diabetes_type2',
 'hx_diabetes_type1',
 'hx_asthma',
 'hx_copd',
 'hx_heart_failure',
 'hx_atrial_fibrillation',
 'hx_ckd',
 'hx_liver_disease',
 'hx_malignancy',
 'hx_obesity',
 'hx_depression',
 'hx_anxiety',
 'hx_dementia',
 'hx_epilepsy',
 'hx_hypothyroi

In [5]:
missing_summary = (
    df.isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["variable", "n_missing"]
missing_summary["pct_missing"] = (missing_summary["n_missing"] / len(df) * 100).round(2)

missing_summary = missing_summary.sort_values("n_missing", ascending=False)

missing_summary[missing_summary["n_missing"] > 0]

,variable,n_missing,pct_missing
25,pulse_pressure,4146,5.18
22,systolic_bp,4146,5.18
23,diastolic_bp,4146,5.18
24,mean_arterial_pressure,4146,5.18
35,shock_index,4146,5.18
66,pp_recalc,4146,5.18
27,respiratory_rate,3067,3.83
28,temperature_c,574,0.72


In [6]:
pain_not_recorded_n = (df["pain_score"] == -1).sum()
pain_not_recorded_pct = (pain_not_recorded_n / len(df) * 100).round(2)

print(f"Pain score = -1: {pain_not_recorded_n} casos ({pain_not_recorded_pct}%)")

Pain score = -1: 11156 casos (13.94%)


In [7]:
df["pain_score"].value_counts(dropna=False).sort_index()

pain_score
-1     11156
 0      1922
 1      4488
 2      4626
 3      7918
 4      8044
 5      8086
 6      7986
 7      8761
 8      5963
 9      5913
 10     5137
Name: count, dtype: int64

In [8]:
# Presión arterial faltante
df["bp_missing_flag"] = (
    df["systolic_bp"].isna() | 
    df["diastolic_bp"].isna()
).astype(int)

# Signos vitales faltantes individuales
df["systolic_bp_missing_flag"] = df["systolic_bp"].isna().astype(int)
df["diastolic_bp_missing_flag"] = df["diastolic_bp"].isna().astype(int)
df["respiratory_rate_missing_flag"] = df["respiratory_rate"].isna().astype(int)
df["temperature_missing_flag"] = df["temperature_c"].isna().astype(int)

# Derivadas faltantes
df["map_missing_flag"] = df["mean_arterial_pressure"].isna().astype(int)
df["pulse_pressure_missing_flag"] = df["pulse_pressure"].isna().astype(int)
df["shock_index_missing_flag"] = df["shock_index"].isna().astype(int)

# Dolor no registrado
df["pain_not_recorded_flag"] = (df["pain_score"] == -1).astype(int)

# Cualquier signo vital faltante relevante
df["any_vital_missing_flag"] = (
    (df["bp_missing_flag"] == 1) |
    (df["respiratory_rate_missing_flag"] == 1) |
    (df["temperature_missing_flag"] == 1)
).astype(int)

# Cualquier missing clínico relevante, incluyendo dolor no registrado
df["any_triage_data_missing_flag"] = (
    (df["any_vital_missing_flag"] == 1) |
    (df["pain_not_recorded_flag"] == 1)
).astype(int)

In [9]:
missing_flags = [
    "bp_missing_flag",
    "systolic_bp_missing_flag",
    "diastolic_bp_missing_flag",
    "respiratory_rate_missing_flag",
    "temperature_missing_flag",
    "map_missing_flag",
    "pulse_pressure_missing_flag",
    "shock_index_missing_flag",
    "pain_not_recorded_flag",
    "any_vital_missing_flag",
    "any_triage_data_missing_flag"
]

missing_flags_summary = pd.DataFrame({
    "flag": missing_flags,
    "n": [df[col].sum() for col in missing_flags]
})

missing_flags_summary["%"] = (
    missing_flags_summary["n"] / len(df) * 100
).round(2)

missing_flags_summary.sort_values("n", ascending=False)

,flag,n,%
10,any_triage_data_missing_flag,17171,21.46
8,pain_not_recorded_flag,11156,13.94
9,any_vital_missing_flag,7331,9.16
2,diastolic_bp_missing_flag,4146,5.18
1,systolic_bp_missing_flag,4146,5.18
0,bp_missing_flag,4146,5.18
6,pulse_pressure_missing_flag,4146,5.18
5,map_missing_flag,4146,5.18
7,shock_index_missing_flag,4146,5.18
3,respiratory_rate_missing_flag,3067,3.83


In [10]:
missing_pattern_cols = [
    "systolic_bp_missing_flag",
    "diastolic_bp_missing_flag",
    "map_missing_flag",
    "pulse_pressure_missing_flag",
    "shock_index_missing_flag",
    "respiratory_rate_missing_flag",
    "temperature_missing_flag",
    "pain_not_recorded_flag"
]

missing_patterns = (
    df[missing_pattern_cols]
    .value_counts()
    .reset_index(name="n")
)

missing_patterns["%"] = (missing_patterns["n"] / len(df) * 100).round(2)

missing_patterns.head(20)

,systolic_bp_missing_flag,diastolic_bp_missing_flag,map_missing_flag,pulse_pressure_missing_flag,shock_index_missing_flag,respiratory_rate_missing_flag,temperature_missing_flag,pain_not_recorded_flag,n,%
0,0,0,0,0,0,0,0,0,62829,78.54
1,0,0,0,0,0,0,0,1,9840,12.30
2,1,1,1,1,1,0,0,0,3092,3.86
3,0,0,0,0,0,1,0,0,2155,2.69
4,1,1,1,1,1,0,0,1,644,0.80
5,0,0,0,0,0,1,0,1,510,0.64
6,0,0,0,0,0,0,1,0,396,0.50
7,1,1,1,1,1,1,0,0,290,0.36
8,0,0,0,0,0,0,1,1,82,0.10
9,1,1,1,1,1,1,0,1,66,0.08


In [11]:
missing_by_esi = (
    df.groupby("triage_acuity")[missing_flags]
    .mean()
    .T
    .round(3) * 100
)

missing_by_esi

triage_acuity,1,2,3,4,5
bp_missing_flag,0.0,0.0,0.0,12.1,11.8
systolic_bp_missing_flag,0.0,0.0,0.0,12.1,11.8
diastolic_bp_missing_flag,0.0,0.0,0.0,12.1,11.8
respiratory_rate_missing_flag,0.0,0.0,0.0,8.9,8.9
temperature_missing_flag,0.0,0.0,0.0,0.0,5.0
map_missing_flag,0.0,0.0,0.0,12.1,11.8
pulse_pressure_missing_flag,0.0,0.0,0.0,12.1,11.8
shock_index_missing_flag,0.0,0.0,0.0,12.1,11.8
pain_not_recorded_flag,0.0,0.0,18.0,17.4,17.1
any_vital_missing_flag,0.0,0.0,0.0,20.0,23.9


In [12]:
missing_by_esi_n = (
    df.groupby("triage_acuity")[missing_flags]
    .sum()
    .T
)

missing_by_esi_n

triage_acuity,1,2,3,4,5
bp_missing_flag,0,0,0,2796,1350
systolic_bp_missing_flag,0,0,0,2796,1350
diastolic_bp_missing_flag,0,0,0,2796,1350
respiratory_rate_missing_flag,0,0,0,2053,1014
temperature_missing_flag,0,0,0,0,574
map_missing_flag,0,0,0,2796,1350
pulse_pressure_missing_flag,0,0,0,2796,1350
shock_index_missing_flag,0,0,0,2796,1350
pain_not_recorded_flag,0,0,5192,4014,1950
any_vital_missing_flag,0,0,0,4607,2724


In [13]:
missing_by_disposition = (
    df.groupby("disposition")[missing_flags]
    .mean()
    .T
    .round(3) * 100
)

missing_by_disposition

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
bp_missing_flag,1.2,0.0,8.1,6.9,7.9,2.2,2.2
systolic_bp_missing_flag,1.2,0.0,8.1,6.9,7.9,2.2,2.2
diastolic_bp_missing_flag,1.2,0.0,8.1,6.9,7.9,2.2,2.2
respiratory_rate_missing_flag,0.8,0.0,6.2,4.9,5.5,1.8,1.1
temperature_missing_flag,0.0,0.0,1.3,0.6,1.2,0.1,0.2
map_missing_flag,1.2,0.0,8.1,6.9,7.9,2.2,2.2
pulse_pressure_missing_flag,1.2,0.0,8.1,6.9,7.9,2.2,2.2
shock_index_missing_flag,1.2,0.0,8.1,6.9,7.9,2.2,2.2
pain_not_recorded_flag,9.6,0.0,16.9,16.9,17.1,12.9,10.8
any_vital_missing_flag,1.9,0.0,14.6,11.6,13.9,3.9,3.3


In [14]:
missing_by_disposition_n = (
    df.groupby("disposition")[missing_flags]
    .sum()
    .T
)

missing_by_disposition_n

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
bp_missing_flag,290,0,3164,190,289,96,117
systolic_bp_missing_flag,290,0,3164,190,289,96,117
diastolic_bp_missing_flag,290,0,3164,190,289,96,117
respiratory_rate_missing_flag,199,0,2401,135,200,76,56
temperature_missing_flag,10,0,489,17,44,6,8
map_missing_flag,290,0,3164,190,289,96,117
pulse_pressure_missing_flag,290,0,3164,190,289,96,117
shock_index_missing_flag,290,0,3164,190,289,96,117
pain_not_recorded_flag,2360,0,6581,468,627,560,560
any_vital_missing_flag,474,0,5688,320,510,168,171


In [15]:
missing_by_mental_status = (
    df.groupby("mental_status_triage")[missing_flags]
    .mean()
    .T
    .round(3) * 100
)

missing_by_mental_status

mental_status_triage,agitated,alert,confused,drowsy,unresponsive
bp_missing_flag,2.7,7.5,2.5,1.9,0.0
systolic_bp_missing_flag,2.7,7.5,2.5,1.9,0.0
diastolic_bp_missing_flag,2.7,7.5,2.5,1.9,0.0
respiratory_rate_missing_flag,2.0,5.5,1.8,1.4,0.0
temperature_missing_flag,0.1,1.2,0.1,0.1,0.0
map_missing_flag,2.7,7.5,2.5,1.9,0.0
pulse_pressure_missing_flag,2.7,7.5,2.5,1.9,0.0
shock_index_missing_flag,2.7,7.5,2.5,1.9,0.0
pain_not_recorded_flag,10.7,16.9,13.0,8.3,1.9
any_vital_missing_flag,4.6,13.3,4.2,3.2,0.0


In [16]:
missing_by_age_group = (
    df.groupby("age_group")[missing_flags]
    .mean()
    .T
    .round(3) * 100
)

missing_by_age_group

age_group,elderly,middle_aged,pediatric,young_adult
bp_missing_flag,5.2,5.2,5.2,5.2
systolic_bp_missing_flag,5.2,5.2,5.2,5.2
diastolic_bp_missing_flag,5.2,5.2,5.2,5.2
respiratory_rate_missing_flag,3.8,3.8,4.0,3.8
temperature_missing_flag,0.7,0.7,0.7,0.8
map_missing_flag,5.2,5.2,5.2,5.2
pulse_pressure_missing_flag,5.2,5.2,5.2,5.2
shock_index_missing_flag,5.2,5.2,5.2,5.2
pain_not_recorded_flag,13.5,14.2,13.8,14.2
any_vital_missing_flag,9.2,9.1,9.2,9.1


In [17]:
missing_by_pediatric = (
    df.groupby("is_pediatric")[missing_flags]
    .mean()
    .T
    .round(3) * 100
)

missing_by_pediatric

is_pediatric,0,1
bp_missing_flag,5.2,5.2
systolic_bp_missing_flag,5.2,5.2
diastolic_bp_missing_flag,5.2,5.2
respiratory_rate_missing_flag,3.8,3.9
temperature_missing_flag,0.7,0.8
map_missing_flag,5.2,5.2
pulse_pressure_missing_flag,5.2,5.2
shock_index_missing_flag,5.2,5.2
pain_not_recorded_flag,14.0,13.9
any_vital_missing_flag,9.1,9.4


In [18]:
missing_by_site = (
    df.groupby("site_id")[missing_flags]
    .mean()
    .T
    .round(3) * 100
)

missing_by_site

site_id,SITE-HEL-01,SITE-HEL-02,SITE-OUL-01,SITE-TMP-01,SITE-TUR-01
bp_missing_flag,5.1,5.5,5.3,5.2,4.9
systolic_bp_missing_flag,5.1,5.5,5.3,5.2,4.9
diastolic_bp_missing_flag,5.1,5.5,5.3,5.2,4.9
respiratory_rate_missing_flag,3.9,3.9,4.1,3.7,3.6
temperature_missing_flag,0.5,0.9,0.7,0.7,0.7
map_missing_flag,5.1,5.5,5.3,5.2,4.9
pulse_pressure_missing_flag,5.1,5.5,5.3,5.2,4.9
shock_index_missing_flag,5.1,5.5,5.3,5.2,4.9
pain_not_recorded_flag,14.2,13.8,13.8,13.9,14.0
any_vital_missing_flag,9.0,9.6,9.5,9.0,8.7


In [19]:
missing_by_shift = (
    df.groupby("shift")[missing_flags]
    .mean()
    .T
    .round(3) * 100
)

missing_by_shift

shift,afternoon,evening,morning,night
bp_missing_flag,5.2,5.2,5.0,5.4
systolic_bp_missing_flag,5.2,5.2,5.0,5.4
diastolic_bp_missing_flag,5.2,5.2,5.0,5.4
respiratory_rate_missing_flag,3.9,3.8,3.8,3.9
temperature_missing_flag,0.6,0.8,0.7,0.7
map_missing_flag,5.2,5.2,5.0,5.4
pulse_pressure_missing_flag,5.2,5.2,5.0,5.4
shock_index_missing_flag,5.2,5.2,5.0,5.4
pain_not_recorded_flag,13.8,14.2,13.9,14.0
any_vital_missing_flag,9.1,9.3,9.0,9.4


In [20]:
missing_by_arrival_mode = (
    df.groupby("arrival_mode")[missing_flags]
    .mean()
    .T
    .round(3) * 100
)

missing_by_arrival_mode

arrival_mode,ambulance,brought_by_family,helicopter,police,transfer,walk-in
bp_missing_flag,5.2,4.9,5.4,5.4,5.5,5.1
systolic_bp_missing_flag,5.2,4.9,5.4,5.4,5.5,5.1
diastolic_bp_missing_flag,5.2,4.9,5.4,5.4,5.5,5.1
respiratory_rate_missing_flag,4.0,4.0,3.5,3.8,4.0,3.7
temperature_missing_flag,0.7,0.7,0.7,0.9,0.8,0.7
map_missing_flag,5.2,4.9,5.4,5.4,5.5,5.1
pulse_pressure_missing_flag,5.2,4.9,5.4,5.4,5.5,5.1
shock_index_missing_flag,5.2,4.9,5.4,5.4,5.5,5.1
pain_not_recorded_flag,13.6,14.3,14.9,14.1,14.2,14.0
any_vital_missing_flag,9.3,9.1,9.1,9.6,9.6,9.0


In [21]:
missing_by_complaint_system = (
    df.groupby("chief_complaint_system")[missing_flags]
    .mean()
    .T
    .round(3) * 100
)

missing_by_complaint_system

chief_complaint_system,ENT,cardiovascular,dermatological,endocrine,gastrointestinal,genitourinary,infectious,musculoskeletal,neurological,ophthalmic,other,psychiatric,respiratory,trauma
bp_missing_flag,5.4,5.1,5.4,5.1,4.6,5.3,5.1,5.5,5.4,5.4,4.9,5.5,4.5,5.4
systolic_bp_missing_flag,5.4,5.1,5.4,5.1,4.6,5.3,5.1,5.5,5.4,5.4,4.9,5.5,4.5,5.4
diastolic_bp_missing_flag,5.4,5.1,5.4,5.1,4.6,5.3,5.1,5.5,5.4,5.4,4.9,5.5,4.5,5.4
respiratory_rate_missing_flag,3.9,3.8,3.8,3.1,3.2,4.2,3.9,3.9,4.1,4.3,3.5,4.0,3.9,4.1
temperature_missing_flag,0.8,0.6,0.6,0.9,0.7,0.7,0.6,0.6,0.8,0.9,0.7,0.7,0.5,0.8
map_missing_flag,5.4,5.1,5.4,5.1,4.6,5.3,5.1,5.5,5.4,5.4,4.9,5.5,4.5,5.4
pulse_pressure_missing_flag,5.4,5.1,5.4,5.1,4.6,5.3,5.1,5.5,5.4,5.4,4.9,5.5,4.5,5.4
shock_index_missing_flag,5.4,5.1,5.4,5.1,4.6,5.3,5.1,5.5,5.4,5.4,4.9,5.5,4.5,5.4
pain_not_recorded_flag,14.6,13.6,14.4,13.9,13.8,14.0,13.3,14.5,13.8,13.3,13.7,14.1,14.0,14.2
any_vital_missing_flag,9.4,8.9,9.3,8.6,8.1,9.7,9.0,9.6,9.7,10.0,8.5,9.6,8.5,9.6


In [22]:
def missing_rate_by_group(df, group_col, flag_cols):
    rows = []
    
    for flag in flag_cols:
        temp = (
            df.groupby(group_col)[flag]
            .agg(["count", "sum", "mean"])
            .reset_index()
        )
        
        temp["flag"] = flag
        temp["pct_missing"] = (temp["mean"] * 100).round(2)
        temp = temp.rename(columns={
            "count": "n_group",
            "sum": "n_missing_or_flag"
        })
        
        rows.append(temp[[group_col, "flag", "n_group", "n_missing_or_flag", "pct_missing"]])
    
    return pd.concat(rows, ignore_index=True).sort_values(
        ["flag", "pct_missing"], ascending=[True, False]
    )

In [23]:
missing_ranking_esi = missing_rate_by_group(
    df=df,
    group_col="triage_acuity",
    flag_cols=missing_flags
)

missing_ranking_esi.head(50)

,triage_acuity,flag,n_group,n_missing_or_flag,pct_missing
54,5,any_triage_data_missing_flag,11398,4197,36.82
53,4,any_triage_data_missing_flag,23020,7782,33.81
52,3,any_triage_data_missing_flag,28921,5192,17.95
50,1,any_triage_data_missing_flag,3222,0,0.00
51,2,any_triage_data_missing_flag,13439,0,0.00
49,5,any_vital_missing_flag,11398,2724,23.90
48,4,any_vital_missing_flag,23020,4607,20.01
45,1,any_vital_missing_flag,3222,0,0.00
46,2,any_vital_missing_flag,13439,0,0.00
47,3,any_vital_missing_flag,28921,0,0.00


In [24]:
missing_ranking_disposition = missing_rate_by_group(
    df=df,
    group_col="disposition",
    flag_cols=missing_flags
)

missing_ranking_disposition.head(80)

,disposition,flag,n_group,n_missing_or_flag,pct_missing
72,discharged,any_triage_data_missing_flag,39028,11229,28.77
74,lwbs,any_triage_data_missing_flag,3656,1050,28.72
73,lama,any_triage_data_missing_flag,2764,728,26.34
75,observation,any_triage_data_missing_flag,4337,703,16.21
76,transferred,any_triage_data_missing_flag,5203,692,13.30
...,...,...,...,...,...
31,lama,temperature_missing_flag,2764,17,0.62
34,transferred,temperature_missing_flag,5203,8,0.15
33,observation,temperature_missing_flag,4337,6,0.14
28,admitted,temperature_missing_flag,24601,10,0.04


In [25]:
missing_ranking_site = missing_rate_by_group(
    df=df,
    group_col="site_id",
    flag_cols=missing_flags
)

missing_ranking_site.head(80)

,site_id,flag,n_group,n_missing_or_flag,pct_missing
51,SITE-HEL-02,any_triage_data_missing_flag,15912,3460,21.74
52,SITE-OUL-01,any_triage_data_missing_flag,15847,3440,21.71
50,SITE-HEL-01,any_triage_data_missing_flag,16161,3480,21.53
53,SITE-TMP-01,any_triage_data_missing_flag,15868,3385,21.33
54,SITE-TUR-01,any_triage_data_missing_flag,16212,3406,21.01
46,SITE-HEL-02,any_vital_missing_flag,15912,1526,9.59
47,SITE-OUL-01,any_vital_missing_flag,15847,1511,9.53
48,SITE-TMP-01,any_vital_missing_flag,15868,1434,9.04
45,SITE-HEL-01,any_vital_missing_flag,16161,1447,8.95
49,SITE-TUR-01,any_vital_missing_flag,16212,1413,8.72


In [26]:
conditions = [
    (df["bp_missing_flag"] == 1) & (df["respiratory_rate_missing_flag"] == 0) & (df["temperature_missing_flag"] == 0),
    (df["bp_missing_flag"] == 0) & (df["respiratory_rate_missing_flag"] == 1) & (df["temperature_missing_flag"] == 0),
    (df["bp_missing_flag"] == 0) & (df["respiratory_rate_missing_flag"] == 0) & (df["temperature_missing_flag"] == 1),
    (df["bp_missing_flag"] == 1) & (df["respiratory_rate_missing_flag"] == 1),
    (df["bp_missing_flag"] == 1) & (df["temperature_missing_flag"] == 1),
    (df["respiratory_rate_missing_flag"] == 1) & (df["temperature_missing_flag"] == 1),
    (df["any_vital_missing_flag"] == 0) & (df["pain_not_recorded_flag"] == 1),
    (df["any_triage_data_missing_flag"] == 0)
]

choices = [
    "01_bp_only_missing",
    "02_rr_only_missing",
    "03_temp_only_missing",
    "04_bp_and_rr_missing",
    "05_bp_and_temp_missing",
    "06_rr_and_temp_missing",
    "07_pain_only_not_recorded",
    "00_no_relevant_missing"
]

df["missingness_pattern_category"] = np.select(
    conditions,
    choices,
    default="99_multiple_missing_pattern"
)

In [27]:
missingness_pattern_summary = (
    df["missingness_pattern_category"]
    .value_counts(dropna=False)
    .rename_axis("missingness_pattern_category")
    .reset_index(name="n")
)

missingness_pattern_summary["%"] = (
    missingness_pattern_summary["n"] / len(df) * 100
).round(2)

missingness_pattern_summary

,missingness_pattern_category,n,%
0,00_no_relevant_missing,62829,78.54
1,07_pain_only_not_recorded,9840,12.30
2,01_bp_only_missing,3736,4.67
3,02_rr_only_missing,2665,3.33
4,03_temp_only_missing,478,0.60
5,04_bp_and_rr_missing,360,0.45
6,05_bp_and_temp_missing,50,0.06
7,06_rr_and_temp_missing,42,0.05


In [28]:
pd.crosstab(
    df["missingness_pattern_category"],
    df["triage_acuity"],
    normalize="index"
).round(3) * 100

triage_acuity,1,2,3,4,5
missingness_pattern_category,,,,,
00_no_relevant_missing,5.1,21.4,37.8,24.3,11.5
01_bp_only_missing,0.0,0.0,0.0,68.4,31.6
02_rr_only_missing,0.0,0.0,0.0,68.0,32.0
03_temp_only_missing,0.0,0.0,0.0,0.0,100.0
04_bp_and_rr_missing,0.0,0.0,0.0,67.2,32.8
05_bp_and_temp_missing,0.0,0.0,0.0,0.0,100.0
06_rr_and_temp_missing,0.0,0.0,0.0,0.0,100.0
07_pain_only_not_recorded,0.0,0.0,52.8,32.3,15.0


In [29]:
pd.crosstab(
    df["missingness_pattern_category"],
    df["disposition"],
    normalize="index"
).round(3) * 100

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
missingness_pattern_category,,,,,,,
00_no_relevant_missing,34.7,0.7,44.2,3.2,4.1,5.8,7.2
01_bp_only_missing,7.1,0.0,75.9,4.6,7.1,2.4,2.9
02_rr_only_missing,6.6,0.0,78.0,4.3,6.8,2.5,1.8
03_temp_only_missing,1.5,0.0,86.0,2.5,8.4,0.6,1.0
04_bp_and_rr_missing,6.1,0.0,80.0,4.7,5.3,1.9,1.9
05_bp_and_temp_missing,6.0,0.0,78.0,4.0,6.0,2.0,4.0
06_rr_and_temp_missing,0.0,0.0,83.3,7.1,2.4,4.8,2.4
07_pain_only_not_recorded,23.3,0.0,56.3,4.1,5.5,5.4,5.3


In [30]:
pd.crosstab(
    df["missingness_pattern_category"],
    df["disposition"],
    normalize="index"
).round(3) * 100

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
missingness_pattern_category,,,,,,,
00_no_relevant_missing,34.7,0.7,44.2,3.2,4.1,5.8,7.2
01_bp_only_missing,7.1,0.0,75.9,4.6,7.1,2.4,2.9
02_rr_only_missing,6.6,0.0,78.0,4.3,6.8,2.5,1.8
03_temp_only_missing,1.5,0.0,86.0,2.5,8.4,0.6,1.0
04_bp_and_rr_missing,6.1,0.0,80.0,4.7,5.3,1.9,1.9
05_bp_and_temp_missing,6.0,0.0,78.0,4.0,6.0,2.0,4.0
06_rr_and_temp_missing,0.0,0.0,83.3,7.1,2.4,4.8,2.4
07_pain_only_not_recorded,23.3,0.0,56.3,4.1,5.5,5.4,5.3


In [31]:
from pathlib import Path

output_dir = Path(r"C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "train_full_missingness_audit_v1.csv"

df.to_csv(output_path, index=False)

print(f"Archivo guardado en: {output_path}")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Archivo guardado en: C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed\train_full_missingness_audit_v1.csv
Filas: 80000
Columnas: 105


In [32]:
df_check = pd.read_csv(output_path)

print(df_check.shape)

df_check[[
    "bp_missing_flag",
    "respiratory_rate_missing_flag",
    "temperature_missing_flag",
    "pain_not_recorded_flag",
    "any_vital_missing_flag",
    "any_triage_data_missing_flag",
    "missingness_pattern_category"
]].head()

(80000, 105)


,bp_missing_flag,respiratory_rate_missing_flag,temperature_missing_flag,pain_not_recorded_flag,any_vital_missing_flag,any_triage_data_missing_flag,missingness_pattern_category
0,0,0,0,0,0,0,00_no_relevant_missing
1,0,0,0,1,0,1,07_pain_only_not_recorded
2,0,0,0,0,0,0,00_no_relevant_missing
3,0,0,0,0,0,0,00_no_relevant_missing
4,0,0,0,0,0,0,00_no_relevant_missing


# Conclusiones análisis de missingness

## Objetivo

Evaluar los datos faltantes formales y codificados antes de definir una estrategia de imputación y modelado.

Se analizaron principalmente:

* Presión arterial sistólica y diastólica.
* Presión arterial media, presión de pulso y shock index.
* Frecuencia respiratoria.
* Temperatura.
* `pain_score = -1` como valor codificado de dolor no registrado.

## Hallazgos principales

### 1. El missingness no parece completamente aleatorio

Los datos faltantes no se distribuyen de forma uniforme en el dataset.

La ausencia de presión arterial, frecuencia respiratoria y temperatura se concentra en pacientes de menor acuidad, especialmente ESI 4 y ESI 5. En cambio, no se observa missingness de estos signos vitales en ESI 1-3.

Esto sugiere que el mecanismo de missingness no es MCAR, sino más probablemente MAR o un patrón operacional/sintético relacionado con baja acuidad.

### 2. La presión arterial faltante parece un fenómeno operacional de baja acuidad

La presión arterial faltante aparece solo en ESI 4-5 y se asocia principalmente a pacientes dados de alta, LWBS o LAMA, sin presencia relevante en fallecidos.

Esto sugiere que la ausencia de PA probablemente no representa gravedad oculta, sino ausencia de medición o documentación en pacientes de menor acuidad.

### 3. La temperatura faltante parece una regla aún más específica

La temperatura faltante se concentra exclusivamente en ESI 5. Esto sugiere un patrón altamente estructurado, posiblemente relacionado con la generación sintética del dataset o con documentación incompleta en casos leves.

### 4. `pain_score = -1` debe tratarse como missing codificado

El valor `pain_score = -1` no debe interpretarse como dolor cero. Representa dolor no registrado o no documentado.

Se creó `pain_not_recorded_flag` para conservar esta información. En etapas posteriores, se recomienda crear una versión limpia de dolor reemplazando `-1` por `NaN` antes de imputar.

### 5. Las derivadas no deben imputarse directamente

Variables como `mean_arterial_pressure`, `pulse_pressure` y `shock_index` dependen de signos vitales primitivos. Por lo tanto, no deben imputarse como si fueran mediciones originales.

La estrategia recomendada es imputar o manejar primero las variables primitivas y luego recalcular las derivadas si corresponde.

## Decisiones metodológicas

* No eliminar filas con datos faltantes.
* Conservar flags de missingness.
* No tratar `pain_score = -1` como valor clínico real.
* No imputar directamente variables derivadas de PA.
* Usar las flags de missingness como señales potencialmente informativas en el modelo.
* Definir la imputación final dentro del pipeline de modelado para evitar leakage.

## Conclusión

El missingness del dataset parece tener significado clínico-operacional y no debe tratarse como simple ruido aleatorio.

La ausencia de ciertos signos vitales parece asociarse a menor acuidad y documentación incompleta, mientras que el dolor no registrado representa una forma de missing codificado.

La estrategia recomendada es conservar estas señales mediante flags, imputar con cautela en etapas posteriores y evaluar si las variables de missingness aportan al modelo predictivo.
